extra code for help

In [ ]:
# Combine two overlapping Harvest exports into one, keeping the newer rows and saving an audit of the overlaps


import pandas as pd
input_dir = "../data/extracts/"

old_file = input_dir + "harvest_kwh_15min_250723-260508.csv"
new_file = input_dir + "harvest_kwh_15min_260504-260713.csv"

output_file = "harvest_kwh_15min_250723-260713.csv"
audit_file = "harvest_kwh_overlap_audit.csv"

# Read both exports
old = pd.read_csv(old_file)
new = pd.read_csv(new_file)

# Clean column names
old.columns = old.columns.str.strip()
new.columns = new.columns.str.strip()

# Make sure the files have the same columns
if set(old.columns) != set(new.columns):
    raise ValueError(
        "The columns do not match.\n"
        f"Only in older file: {sorted(set(old.columns) - set(new.columns))}\n"
        f"Only in newer file: {sorted(set(new.columns) - set(old.columns))}"
    )

# Put newer file columns in the same order as the older file
new = new[old.columns]

# Parse timestamps
old["datetime"] = pd.to_datetime(old["datetime"], errors="coerce")
new["datetime"] = pd.to_datetime(new["datetime"], errors="coerce")

if old["datetime"].isna().any():
    raise ValueError(
        f"Older file contains {old['datetime'].isna().sum()} invalid timestamps."
    )

if new["datetime"].isna().any():
    raise ValueError(
        f"Newer file contains {new['datetime'].isna().sum()} invalid timestamps."
    )

# Long-format Harvest data: one unique row per meter and timestamp
if "meter_name" in old.columns:
    old["meter_name"] = old["meter_name"].astype("string").str.strip()
    new["meter_name"] = new["meter_name"].astype("string").str.strip()
    duplicate_key = ["datetime", "meter_name"]
else:
    # Use this only if these are wide-format files
    duplicate_key = ["datetime"]

# Record the source so overlaps can be audited
old["_source_file"] = old_file
new["_source_file"] = new_file

# Older first, newer second
all_rows = pd.concat([old, new], ignore_index=True)

# Save all overlapping rows for review
duplicate_mask = all_rows.duplicated(
    subset=duplicate_key,
    keep=False
)

overlap_audit = (
    all_rows.loc[duplicate_mask]
    .sort_values(duplicate_key + ["_source_file"])
)

overlap_audit.to_csv(
    audit_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

# Keep the last duplicate, so the newer export wins
combined = (
    all_rows
    .drop_duplicates(subset=duplicate_key, keep="last")
    .drop(columns="_source_file")
    .sort_values(duplicate_key)
    .reset_index(drop=True)
)

# Final validation
assert not combined.duplicated(subset=duplicate_key).any()

combined.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print(f"Older rows:           {len(old):,}")
print(f"Newer rows:           {len(new):,}")
print(f"Rows before dedupe:   {len(all_rows):,}")
print(f"Rows removed:         {len(all_rows) - len(combined):,}")
print(f"Combined rows:        {len(combined):,}")
print(f"First timestamp:      {combined['datetime'].min()}")
print(f"Last timestamp:       {combined['datetime'].max()}")
print(f"Combined file saved:  {output_file}")
print(f"Overlap audit saved:  {audit_file}")

In [ ]:
# finding min and max datetime in the data file

import pandas as pd

var_file = input_dir + 'harvest_kwh_15min_250723-260713.csv'

df = pd.read_csv(var_file, usecols=["datetime"])
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

min_dt = df["datetime"].min()
max_dt = df["datetime"].max()

print("min datetime:", min_dt)
print("max datetime:", max_dt)